# Fixed-state dependence of the velocity spectrum on component population

**Computed at n=27 (512^3), using the sparse compiler and the FDM reconstruction.**
All settings and both component states remain fixed while chi changes: beta=0, k_d=15, target P0=0.1, phase seeds 2025/2026, locality_radius=3, tail_tolerance=1e-12.

The same separately normalized profiles are reused for every case. Define $d_s=|\varphi_s|^2$ and $\boldsymbol{j}_s=\operatorname{Im}(\varphi_s^*D_h\varphi_s)$.
The velocity is evaluated through the exact recombination
$\boldsymbol{u}_\varTheta=
\frac{\varTheta\boldsymbol{j}_+ +(1-\varTheta)\boldsymbol{j}_-}
     {\varTheta d_+ +(1-\varTheta)d_-}.$
No density floor or velocity filtering is applied. All calculations use double precision; display-only vorticity and helicity arrays are stored in float32.

The spectrum is the **unweighted, unit-width rounded shell sum**, with the FFT
normalized by N^3. The common primary fit uses every positive integer shell
from k=15 to k=256, with unweighted OLS in log(E_k) versus log(k).
Gamma is defined by E_k proportional to k^(-gamma). The ten chi values include
an exploratory point at 0.02 and a refined point at 0.02 targeting gamma=2.

**Figures:** top row, three vorticity isosurfaces at |omega|=300 colored by
h=u dot omega with limits [-2000,2000]. The camera, box, and color scale are
identical. Only smooth shading is enabled; surface coordinates and helicity
are not smoothed. Bottom left, raw spectra and primary fits, restricted to
complete shells k<=N/2. Bottom right, all ten fitted exponents. Gray bars span
results from [15,256], [15,128], [30,256], [30,200], and [20,200]. They show
fit-range sensitivity, **not** ensemble uncertainty or confidence intervals.

The study uses one fixed phase pair. It establishes an empirical parameter
response for this fixed family, without implying a universal chi-to-gamma law
or robustness across independently resampled phases.

Run all cells to reproduce the figure from the saved results. Set
`RUN_SIMULATIONS=True` in the optional cell to regenerate the expensive data
and surfaces. The large component cache is reused automatically.


In [1]:
from pathlib import Path
import importlib.util
import sys

RUN_SIMULATIONS = False

if RUN_SIMULATIONS:
    root = next(p for p in (Path.cwd(), *Path.cwd().parents)
                if (p / "setup_plot.py").is_file())
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    work = root / "outputs/chi_scan_n27"
    spec = importlib.util.spec_from_file_location("chi_scan", work / "chi_scan.py")
    scan = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(scan)
    scan.prepare_components()
    density, currents = scan.load_components()
    shells, symmetry = scan.make_shells()
    for chi in scan.CHIS + [0.02, 0.01947]:
        scan.evaluate_chi(chi, density, currents, shells, symmetry)
    scan.write_summary()
    del density, currents, shells, symmetry

    spec = importlib.util.spec_from_file_location("render_cases", work / "render_cases.py")
    render = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(render)
    for chi in [0.5, 0.01947, 0.006]:
        render.render_case(chi)
else:
    print("Using the saved fixed-state scan and vorticity renders.")


Using the saved fixed-state scan and vorticity renders.


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedLocator, NullFormatter
from PIL import Image

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "setup_plot.py").is_file()
     and (p / "outputs/chi_scan_n27/data/chi_scan_summary.csv").is_file()), None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook inside the QuanEncodeTurb project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from setup_plot import setup_plot
from geoencode_turb.visualization import cmap_BuRd

setup_plot(fontsize=8, fonttype="sans-serif", dpi=150)
mpl.rcParams.update({
    "axes.labelsize":8, "axes.titlesize":8, "xtick.labelsize":8,
    "ytick.labelsize":8, "legend.fontsize":8, "savefig.bbox":None,
})
WORK = PROJECT_ROOT / "outputs/chi_scan_n27"
DATA_DIR = WORK / "data"
SURFACE_DIR = WORK / "figures"
OUTPUT_DIR = PROJECT_ROOT / "figures"
OUTPUT_DIR.mkdir(exist_ok=True)
WIDTH_CM, HEIGHT_CM = 17.9, 13.2
SELECTED_CHIS = [0.5, 0.02, 0.006]
COLORS = ["#CE5044", "#86B6A4", "#7AA7D3"]
FACES = ["#FFE9F1", "#DFEDDC", "#DDE9F5"]
MARKERS = ["o", "s", "d"]


def tag(chi):
    return format(float(chi), ".10g").replace(".", "p").replace("-", "m")


summary = pd.read_csv(DATA_DIR / "chi_scan_summary.csv").sort_values("chi")
selected = []
for chi in SELECTED_CHIS:
    record = json.loads((DATA_DIR / f"spectrum_chi_{tag(chi)}.json").read_text())
    spectrum = pd.read_csv(DATA_DIR / f"spectrum_chi_{tag(chi)}.csv")
    image = Image.open(SURFACE_DIR / f"vorticity_chi_{tag(chi)}.png").convert("RGB")
    selected.append({"chi":chi,"record":record,"spectrum":spectrum,"image":image})

# Export every fit range, with the primary interval identified explicitly.
fit_rows = []
for path in DATA_DIR.glob("spectrum_chi_*.json"):
    record = json.loads(path.read_text())
    fit_rows.append({"chi":record["chi"],"k_min":15,"k_max":256,
                     "primary":True, **{key:record[key] for key in
                     ["slope","gamma","r_squared","slope_regression_se","log_rms_residual"]}})
    for fit in record["alternative_ranges"]:
        fit_rows.append({"chi":record["chi"],"primary":False,**fit})
fit_table = pd.DataFrame(fit_rows).sort_values(["chi","k_min","k_max"])
fit_table.to_csv(DATA_DIR / "all_fit_ranges.csv",index=False)
selected_table = pd.DataFrame([entry["record"] for entry in selected]).drop(columns="alternative_ranges")
selected_table.to_csv(DATA_DIR / "selected_cases.csv",index=False)
print(f"Loaded {len(summary)} chi values on the fixed 512^3 grid.")
print(selected_table[["chi","slope","r_squared","range_slope_min","range_slope_max"]].to_string(index=False))


Loaded 10 chi values on the fixed 512^3 grid.
    chi     slope  r_squared  range_slope_min  range_slope_max
0.50000 -2.695769   0.997865        -2.816746        -2.686774
0.01947 -2.000063   0.994050        -2.119425        -1.844288
0.00600 -1.645698   0.994178        -1.748339        -1.528011


In [ ]:
cm = 1 / 2.54
fig = plt.figure(figsize=(WIDTH_CM*cm, HEIGHT_CM*cm))


def add_axes_cm(left, bottom, width, height):
    return fig.add_axes([left/WIDTH_CM,bottom/HEIGHT_CM,width/WIDTH_CM,height/HEIGHT_CM])


def style_axes(ax, letter):
    ax.grid(color="k",linestyle="-",linewidth=0.5,alpha=0.1)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)
    ax.tick_params(axis="x",which="major",direction="out",length=3,width=0.5,pad=1.5)
    ax.tick_params(axis="y",which="major",direction="out",length=3,width=0.5,pad=1)
    ax.tick_params(which="minor",direction="out",length=1.5,width=0.5)
    ax.text(-0.16,1.025,rf"\textbf{{{letter}}}",transform=ax.transAxes,
            fontsize=8,ha="left",va="bottom")


# Use one crop rectangle for all panels so the spatial magnification is identical.
boxes = []
for entry in selected:
    array = np.asarray(entry["image"])
    y,x = np.nonzero(np.any(array < 245, axis=2))
    boxes.append((x.min(),y.min(),x.max()+1,y.max()+1))
pad = 12
crop = (max(0,min(b[0] for b in boxes)-pad),
        max(0,min(b[1] for b in boxes)-pad),
        min(selected[0]["image"].width,max(b[2] for b in boxes)+pad),
        min(selected[0]["image"].height,max(b[3] for b in boxes)+pad))

for index, entry in enumerate(selected):
    ax = add_axes_cm(0.25+index*5.9,7.05,5.6,5.6)
    ax.imshow(entry["image"].crop(crop),interpolation="none")
    ax.set_axis_off()
    ax.set_title(rf"$\varTheta={entry['chi']:g}$",fontsize=8,pad=5,y=1.0)
    ax.text(-0.015,1.02,rf"\textbf{{{'abc'[index]}}}",transform=ax.transAxes,
            fontsize=8,ha="left",va="bottom")

# Shared helicity-density scale: the same limits as every VTK render.
cax = add_axes_cm(5.45,6.62,7.0,0.18)
colorbar = fig.colorbar(
    mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(-2000,2000),cmap=cmap_BuRd),
    cax=cax,orientation="horizontal",ticks=[-2000,-1000,0,1000,2000],
)
colorbar.outline.set_linewidth(0.5)
colorbar.ax.tick_params(length=2,width=0.5,pad=1,labelsize=8)
fig.text(12.7/WIDTH_CM,6.71/HEIGHT_CM,
         r"$h=\boldsymbol{u}\cdot\boldsymbol{\omega}$",
         fontsize=8,ha="left",va="center")

# d. Raw, unweighted velocity spectra and OLS fits on the common interval.
ax_spectrum = add_axes_cm(1.35,1.05,6.9,4.65)
style_axes(ax_spectrum,"d")
ax_spectrum.set_xscale("log")
ax_spectrum.set_yscale("log")
ax_spectrum.axvspan(15,256,color="#E3E3E3",alpha=0.55,zorder=0)
handles = []
for entry,color in zip(selected,COLORS):
    frame=entry["spectrum"]
    valid=(frame["k"]>=1)&(frame["k"]<=256)&(frame["Ek"]>0)
    line,=ax_spectrum.plot(frame.loc[valid,"k"],frame.loc[valid,"Ek"],color=color,lw=1,
                          label=rf"$\varTheta={entry['chi']:g}$")
    handles.append(line)
    record=entry["record"]
    k_fit=np.geomspace(15,256,120)
    ax_spectrum.plot(k_fit,np.exp(record["intercept_ln"])*k_fit**record["slope"],
                     color=color,lw=0.65,ls="--",dashes=(3,2))
handles.append(Line2D([],[],color="gray",lw=0.65,ls="--",label="Fit"))
ax_spectrum.set_xlim(1,256)
ax_spectrum.set_ylim(1e-4,10)
ax_spectrum.set_xlabel(r"$k$",labelpad=0)
ax_spectrum.set_ylabel(r"$E_k$",rotation=0,labelpad=6)
legend=ax_spectrum.legend(handles=handles,loc="lower left",fontsize=8,frameon=True,
                         fancybox=False,framealpha=0.9,edgecolor="#D0D0D0",
                         labelspacing=0.2,borderpad=0.3,handlelength=1.7,handletextpad=0.5)
legend.get_frame().set_linewidth(0.5)

# e. Gamma is defined by E_k ~ k^(-gamma).
ax_trend = add_axes_cm(10.2,1.05,7.35,4.65)
style_axes(ax_trend,"e")
ax_trend.set_xscale("log")
x=summary["chi"].to_numpy()
y=summary["gamma"].to_numpy()
lower=-summary["range_slope_max"].to_numpy()
upper=-summary["range_slope_min"].to_numpy()
ax_trend.errorbar(x,y,yerr=[y-lower,upper-y],color="#777777",lw=0.8,
                  marker="o",markersize=3.5,markerfacecolor="white",markeredgewidth=0.7,
                  ecolor="#AAAAAA",elinewidth=0.6,capsize=2,capthick=0.6,zorder=2)
for gamma,label in [(1,r"$1$"),(5/3,r"$5/3$"),(2,r"$2$"),(8/3,r"$8/3$")]:
    ax_trend.axhline(gamma,color="#B8B8B8",lw=0.5,ls="--",zorder=0)
for entry,color,face,marker in zip(selected,COLORS,FACES,MARKERS):
    ax_trend.plot(entry["chi"],entry["record"]["gamma"],marker=marker,ls="none",
                  markersize=5,markerfacecolor=face,markeredgecolor=color,
                  markeredgewidth=0.8,zorder=4)
ax_trend.set_xlim(7e-4,0.7)
ax_trend.set_ylim(0.95,2.95)
ax_trend.set_xticks([1e-3,1e-2,1e-1])
ax_trend.set_yticks([1,5/3,2,8/3])
ax_trend.set_yticklabels([r"$1$",r"$5/3$",r"$2$",r"$8/3$"])
ax_trend.xaxis.set_minor_formatter(NullFormatter())
ax_trend.set_xlabel(r"Relative component population, $\varTheta$",labelpad=1)
ax_trend.set_ylabel(r"$\gamma$",rotation=0,labelpad=7)
ax_trend.text(0.035,0.94,r"$E_k\propto k^{-\gamma}$",transform=ax_trend.transAxes,
              fontsize=8,ha="left",va="top")
ax_trend.text(0.035,0.80,r"Bars: fit-range variation",transform=ax_trend.transAxes,
              fontsize=8,ha="left",va="top",color="#555555")

# Numeric slopes are stated on the corresponding isosurface panels.
for index, entry in enumerate(selected):
    fig.text((0.25+index*5.9+2.8)/WIDTH_CM,6.94/HEIGHT_CM,
             rf"$\gamma_{{\mathrm{{fit}}}}={entry['record']['gamma']:.3f}$",
             fontsize=8,ha="center",va="center")

fig.canvas.draw()
renderer=fig.canvas.get_renderer()
for ax in [ax_spectrum,ax_trend,cax]:
    bbox=ax.get_tightbbox(renderer).transformed(fig.transFigure.inverted())
    if bbox.x0<0 or bbox.y0<0 or bbox.x1>1 or bbox.y1>1:
        raise RuntimeError(f"An axis extends outside the canvas: {bbox}")


# 
fig.savefig(OUTPUT_DIR / "chi_dependence_n27.pdf", format="pdf", transparent=True, bbox_inches=None, dpi=600)